In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import configs
import ddpm_class
import FM_class
import evaluate
import plot_func


In [ ]:
cddpm = ddpm_class.cDDPM(save_int=80000, have_rho=False, cddpm_name="cddpm")
cddpm_have_rho = ddpm_class.cDDPM(save_int=80000, have_rho=True, cddpm_name="cddpm_rho")
tddpm = ddpm_class.tDDPM(save_int=80000, tddpm_name="tddpm")

tFM = FM_class.tFM(save_int=80000, tFM_name="tFM")
cFM = FM_class.cFM(save_int=80000, have_rho=False, cFM_name="cFM")
cFM_have_rho = FM_class.cFM(save_int=80000, have_rho=True, cFM_name="cFM_rho")

In [ ]:
cddpm.train(40000)

In [ ]:
cddpm_have_rho.train(40000)

In [ ]:
epoches = 80000
cFM.train(epoches)
cFM_have_rho.train(epoches)

In [ ]:
cFM.train( 80000)

In [ ]:
tddpm.load_ckpt('./saved_model/tddpm_it_80000.pth',)
cddpm.load_ckpt('./saved_model/cddpm_it_80000.pth')
cddpm_have_rho.load_ckpt('./saved_model/cddpm_rho_it_80000.pth')
tFM.load_ckpt('./saved_model/tFM_it_80000.pth',)
cFM.load_ckpt('./saved_model/cFM_it_80000.pth')
cFM_have_rho.load_ckpt('./saved_model/cFM_rho_it_80000.pth')

In [ ]:
from dataset import generate_data
import tqdm
def unfold_batch(model, y):
    batch_size = y.shape[0]
    with torch.no_grad():
        unfolded_part = model.sampling(batch_size, y)

    return unfolded_part

def eval_models(model_list, batch_num, require_rho_list, batch_size = None, rho_list = None):
    if rho_list is None:
        rho_list = np.linspace(-1,1,201)

    if batch_size is None:
        batch_size = configs.batch_size

    MSE_record = np.zeros([len(model_list), len(rho_list)])
    SWD_record = np.zeros([len(model_list), len(rho_list)])

    for rho_index in tqdm.tqdm(range(len(rho_list))):
        rho = rho_list[rho_index]
        unfolded = [[] for _ in range(len(model_list))]
        x_record = []
        for i in range(batch_num):
            x_batch_B2, y_batch_B2, rho = generate_data(batch_size, rho = rho)
            x_batch_B2 = torch.tensor(x_batch_B2).to(configs.device).to(torch.float32)
            y_batch_B2 = torch.tensor(y_batch_B2).to(configs.device).to(torch.float32)

            for model_index in range(len(model_list)):
                if require_rho_list[model_index]:
                    rho_tensor = torch.full((y_batch_B2.shape[0], 1), rho, dtype=torch.float32, device=configs.device)
                    y_batch_B2_cat = torch.cat([y_batch_B2, rho_tensor], dim=1)
                else:
                    y_batch_B2_cat = y_batch_B2

                model = model_list[model_index]
                unfolded_part = unfold_batch(model, y_batch_B2_cat)
                unfolded[model_index].append(unfolded_part)

            x_record.append(x_batch_B2)

        x_record = torch.cat(x_record)

        for model_index in range(len(model_list)):
            # print(unfolded[model_index].__len__())
            unfolded_compute = torch.cat(unfolded[model_index])
            MSE_record[model_index, rho_index] = np.mean(np.linalg.norm(x_record.cpu().numpy() - unfolded_compute.cpu().numpy(), axis=1) ** 2)
            SWD_record[model_index, rho_index] = plot_func.compute_SWD(x_record.cpu().numpy(), unfolded_compute.cpu().numpy(), sample_size= None)
    print(MSE_record)


In [ ]:
eval_models([tFM], 10, [False], batch_size = None, rho_list = [0])

In [ ]:
from evaluate_nosave import eval_models, plot_models
# eval_models([tddpm, cddpm, cddpm_have_rho], 10, [False, False, True])
# plot_models([tddpm, cddpm, cddpm_have_rho, tFM, cFM, cFM_have_rho], 10, [False, False, True, False, False, True], save=True, rho_list=[0.9])
# plot_models([ tFM], 10, [False], save=False, rho_list=[0.9])
plot_models([tFM, cFM, cFM_have_rho], 10, [ False, False, True], save=True, rho_list=[0.9])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

data = np.load('./output/test_record.npz')
MSE_data = data['MSE_record']
SWD_data = data['SWD_record']
model_name = ['tddpm', 'cddpm', 'cddpm_have_rho', 'tFM', 'cFM', 'cFM_have_rho']
x = np.linspace(-1,1,201)

SWD_omni = np.load('./output/omnifold_SWD.npy')
SWD_omni_combine = np.load('./output/omnifold_combine_SWD.npy')
SB_SWD = np.load('./output/SB_SWD.npy')
def plot_data(input_data, model_name, x, title, ylabel, save_name = None, SWD_omni = None, Omni_name = ('omnifold_best', 'omnifold_combine', 'SB')):
    for i in range(len(model_name)):
        plt.plot(x, input_data[i], label=model_name[i])
    if SWD_omni is not None:
        for j in range(len(SWD_omni)):
            plt.plot(x, SWD_omni[j], label=Omni_name[j])

    plt.axvspan(-0.75, -0.25, color='grey', alpha=0.5)
    plt.axvspan(0.25, 0.75, color='grey', alpha=0.5)
    plt.grid()
    plt.legend()
    plt.xlabel(r'$\rho$')
    plt.ylabel(ylabel)
    plt.title(title)
    if save_name is not None:
        plt.savefig('./fig/' + save_name + '.png', bbox_inches='tight', dpi=300)
    plt.show()

plot_data(MSE_data, model_name, x, title='MSE_record', ylabel="MSE", save_name='MSE_record')
plot_data(SWD_data, model_name, x, title='SWD_record', ylabel="SWD", save_name='SWD_record', SWD_omni = (SWD_omni, SWD_omni_combine, SB_SWD))
# plt.plot(SBSWD)





In [ ]:
import numpy as np
import matplotlib.pyplot as plt
color_list = ['b','g','r','black','orange','brown','k']
data = np.load('./output/test_record.npz')
MSE_data = data['MSE_record']
SWD_data = data['SWD_record']
model_name = ['tddpm', 'cddpm', 'cddpm_have_rho', 'EI-FM', 'cFM', 'cFM-$\\gamma$']
x = np.linspace(-1,1,201)

SWD_omni = np.load('./output/omnifold_SWD.npy')
SWD_omni_combine = np.load('./output/omnifold_combine_SWD.npy')
SB_SWD = np.load('./output/SB_SWD.npy')
def plot_data(input_data, model_name, x, title, ylabel, save_name = None, SWD_omni = None, Omni_name = ('Omnifold-best', 'Omnifold-combine', 'SBUnfold'), ylim = None, fontsize = 16):
    for k in range(len(model_name)-3):
        i=k+3
        plt.plot(x, input_data[i], label=model_name[i], color=color_list[k])
    if SWD_omni is not None:
        for j in range(len(SWD_omni)):
            plt.plot(x, SWD_omni[j], label=Omni_name[j], color=color_list[j+3])
    # print(input_data[:,191])
    plt.axvspan(-0.75, -0.25, color='grey', alpha=0.5)
    plt.axvspan(0.25, 0.75, color='grey', alpha=0.5)
    plt.grid()
    plt.legend(fontsize=12)
    plt.xlabel(r'$\gamma$', fontsize=fontsize)
    plt.ylabel(ylabel, fontsize=fontsize)
    plt.xticks([-1,-0.5,0,0.5,1],fontsize=fontsize)
    plt.yticks(fontsize=fontsize)
    # plt.title(title)
    plt.xlim([-1,1])
    if ylim is not None:
        plt.ylim(ylim)
    if save_name is not None:
        plt.savefig('./fig/' + save_name + '.png', bbox_inches='tight', dpi=300)
    plt.show()

plot_data(MSE_data, model_name, x, title='MSE_record', ylabel="MSE", save_name='MSE_record')
plot_data(SWD_data, model_name, x, title='SWD_record', ylabel="SWD", save_name='SWD_record', SWD_omni = (SWD_omni, SWD_omni_combine, SB_SWD), ylim=[0,0.1])
# plt.plot(SBSWD)



